In [0]:
import json
import re
import unicodedata
from collections import defaultdict
import spacy

# ---------------------------------------------------------
# 1. DATEI LADEN
# ---------------------------------------------------------
input_file = "../../data/registers/register-abhandlungen.json"

try:
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"--> {len(data)} Einträge geladen.")
except FileNotFoundError:
    print(f"Fehler: Datei unter '{input_file}' wurde nicht gefunden.")
    exit()

# ---------------------------------------------------------
# 2. SPACY & FILTER VORBEREITEN
# ---------------------------------------------------------
nlp_de = spacy.load("de_core_news_sm")
nlp_fr = spacy.load("fr_core_news_sm")
nlp_la = spacy.blank("la")

SPACY_STOP_WORDS = (
    nlp_de.Defaults.stop_words | 
    nlp_fr.Defaults.stop_words | 
    nlp_la.Defaults.stop_words
)

KNOWN_NAME_VARIANTS = set()

def extract_names_from_json(entries: list[dict]) -> set[str]:
    names = set()
    for entry in entries:
        author_str = entry.get("author", "")
        if author_str:
            raw_names = re.findall(r'\b[a-zA-ZäöüßÄÖÜàâæçéèêëîïôœùûüÿÀÂÆÇÉÈÊËÎÏÔŒÙÛÜŸ]+\b', author_str)
            for name in raw_names:
                names.add(name.lower())
    return names

json_author_names = extract_names_from_json(data)

def extract_keywords(text: str) -> set[str]:
    doc = nlp_de(text)
    spacy_names = {word.lower() for ent in doc.ents if ent.label_ in ("PER", "PERSON") for word in re.findall(r'\b\w+\b', ent.text)}
    
    all_names_to_ignore = json_author_names | spacy_names | KNOWN_NAME_VARIANTS
    
    # Lemmatisierung: unterschiedliche Wortformen (z.B. "Versuche"/"Versuchen"/
    # "Versuchs") werden ueber token.lemma_ auf ihre Grundform ("Versuch")
    # zusammengefuehrt, damit sie im Register nicht als separate Begriffe
    # auftauchen.
    keywords = set()
    for token in doc:
        if not token.is_alpha:
            continue
        word_lower = token.text.lower()
        lemma = token.lemma_
        if (word_lower not in SPACY_STOP_WORDS and 
            word_lower not in all_names_to_ignore and 
            len(lemma) >= 3):
            keywords.add(lemma.capitalize())
            
    return keywords

def get_base_letter(char: str) -> str:
    normalized = unicodedata.normalize("NFD", char)
    base_char = "".join(c for c in normalized if unicodedata.category(c) != "Mn")
    return base_char.upper()

# ---------------------------------------------------------
# 3. REGISTER ERSTELLEN
# ---------------------------------------------------------
nested_index = defaultdict(lambda: defaultdict(list))

for entry in data:
    title_text = entry.get("title", "")
    keywords = extract_keywords(title_text)

    for term in keywords:
        first_letter = get_base_letter(term[0])
        
        # Das Abhandlungs-Objekt (entry) wird 1:1 übernommen
        # (enthaelt bereits "anhang" und "textbeziehungen" aus register-abhandlungen)
        nested_index[first_letter][term].append(entry)

# Alphabetische Sortierung
sorted_grouped_index = {}
for letter in sorted(nested_index.keys()):
    sorted_terms = dict(sorted(nested_index[letter].items()))
    sorted_grouped_index[letter] = sorted_terms

# ---------------------------------------------------------
# 4. OUTPUT SPEICHERN
# ---------------------------------------------------------
output_file = "../../data/registers/register-begriffe.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sorted_grouped_index, f, ensure_ascii=False, indent=2)

print(f"--> Tiefes, nach Buchstaben sortiertes Begriff-Register gespeichert unter: '{output_file}'")


--> 8410 Einträge geladen.
--> Tiefes, nach Buchstaben sortiertes Begriff-Register gespeichert unter: '../../data/registers/register-begriffe.json'
